In [1]:
import requests
import pandas as pd
from urllib.parse import quote

GDELT_DOC_ENDPOINT = "https://api.gdeltproject.org/api/v2/doc/doc"

def _gdelt_timeline(query: str, mode: str, start: str, end: str, smooth: int = 1) -> pd.DataFrame:
    """
    query: GDELT query string (supports boolean, phrases, operators)
    mode: "TimelineVolRaw" or "TimelineTone" (case-insensitive in practice)
    start/end: YYYYMMDDHHMMSS (UTC-ish). Example: 20240101000000
    """
    params = {
        "query": query,
        "mode": mode,
        "format": "json",
        "startdatetime": start,
        "enddatetime": end,
        "timelinesmooth": smooth,
    }
    r = requests.get(GDELT_DOC_ENDPOINT, params=params, timeout=60)
    r.raise_for_status()
    j = r.json()

    # Timeline results usually live under j["timeline"][0]["data"]
    # We'll be defensive because fields can vary by mode.
    timeline = j.get("timeline", [])
    if not timeline:
        return pd.DataFrame(columns=["date", "value"])

    data = timeline[0].get("data", [])
    out = pd.DataFrame(data)

    # GDELT uses slightly different keys depending on mode; normalize to ["date","value"].
    # Common patterns: date, value OR date, tone OR date, count
    if "value" in out.columns:
        out = out.rename(columns={"value": "value"})
    elif "tone" in out.columns:
        out = out.rename(columns={"tone": "value"})
    elif "count" in out.columns:
        out = out.rename(columns={"count": "value"})
    else:
        # fall back: pick the first numeric column
        num_cols = [c for c in out.columns if c != "date"]
        if num_cols:
            out = out.rename(columns={num_cols[0]: "value"})
        else:
            out["value"] = pd.NA

    out["date"] = pd.to_datetime(out["date"])
    out = out[["date", "value"]].sort_values("date")
    return out


def gdelt_stock_news_features(
    ticker: str,
    company_name: str | None,
    start: str,
    end: str,
    *,
    smooth: int = 1,
) -> pd.DataFrame:
    """
    Returns columns: [ticker, date, news_count, avg_tone]
    start/end are YYYYMMDDHHMMSS (e.g., "20240101000000")
    """
    # Query strategy:
    # - If you have company_name, include it as an exact phrase.
    # - Also include ticker as a token, but beware: tickers can collide with common words (e.g., "CAT").
    # You can tighten later with domain filters, "title:" constraints, etc.
    if company_name:
        q = f'("{company_name}" OR {ticker})'
    else:
        q = ticker

    vol = _gdelt_timeline(q, "TimelineVolRaw", start, end, smooth=smooth).rename(columns={"value": "news_count"})
    tone = _gdelt_timeline(q, "TimelineTone", start, end, smooth=smooth).rename(columns={"value": "avg_tone"})

    df = pd.merge(vol, tone, on="date", how="outer").sort_values("date")
    df.insert(0, "ticker", ticker)
    return df


# Example:
# features = gdelt_stock_news_features("MSFT", "Microsoft", "20240101000000", "20241231235959")
# features.head()


In [2]:
out = _gdelt_timeline(query = "Abbott Laboratories", mode="TimelineVolRaw", start = 20250701000000, end = 20250801000000, smooth = 1)

HTTPError: 429 Client Error: Too Many Requests for url: https://api.gdeltproject.org/api/v2/doc/doc?query=Abbott+Laboratories&mode=TimelineVolRaw&format=json&startdatetime=20250701000000&enddatetime=20250801000000&timelinesmooth=1

In [ ]:
metadata = pd.read_csv("data/sp500_metadata.csv", encoding='cp1252')
metadata

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",3/4/1957,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",7/26/2017,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",3/4/1957,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",12/31/2012,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",7/6/2011,1467373,1989
...,...,...,...,...,...,...,...,...
498,XYL,Xylem Inc.,Industrials,Industrial Machinery & Supplies & Components,"White Plains, New York",11/1/2011,1524472,2011
499,YUM,Yum! Brands,Consumer Discretionary,Restaurants,"Louisville, Kentucky",10/6/1997,1041061,1997
500,ZBRA,Zebra Technologies,Information Technology,Electronic Equipment & Instruments,"Lincolnshire, Illinois",12/23/2019,877212,1969
501,ZBH,Zimmer Biomet,Health Care,Health Care Equipment,"Warsaw, Indiana",8/7/2001,1136869,1927
